\Feature Engineering

In [21]:
import pandas as pd
import numpy as np
import sys, os
import warnings
warnings.filterwarnings("ignore")
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler


In [16]:

src_path = os.path.abspath("../src")
if src_path not in sys.path:
    sys.path.append(src_path)


%load_ext autoreload
%autoreload 2

SAMPLE_DIR = r"D:\projects\Healthcare\data\raw_sample"


from data_cleaning import clean_dataset

df_raw = pd.read_csv(os.path.join(SAMPLE_DIR, "stroke_sliced.csv"))
df, clean_report = clean_dataset(df_raw)

print("Cleaned data ready:", df.shape)
print("Target: Diagnosis")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


2026-07-02 16:50:11,370 | data_cleaning | INFO | Cleaning started. Input shape: (15000, 22)
2026-07-02 16:50:11,502 | data_cleaning | INFO | Filled 'Symptoms' with mode (16.7% missing)
2026-07-02 16:50:11,571 | data_cleaning | INFO | Removed 0 duplicate rows
2026-07-02 16:50:11,596 | data_cleaning | INFO | Skipped 'Hypertension' for outliers (2 unique values)
2026-07-02 16:50:11,599 | data_cleaning | INFO | Skipped 'Heart Disease' for outliers (2 unique values)
2026-07-02 16:50:11,610 | data_cleaning | INFO | Skipped 'Stroke History' for outliers (2 unique values)
2026-07-02 16:50:11,614 | data_cleaning | INFO | Cleaning finished. Output shape: (15000, 22)


Cleaned data ready: (15000, 22)
Target: Diagnosis


In [17]:
def engineer_features(df, target_col, id_threshold=0.95, cardinality_threshold=50):
   
    df = df.copy()
    n_rows = len(df)
    report = {"dropped": {}, "datetime_expanded": []}

    for col in list(df.columns):
        if col == target_col:
            continue

        n_unique = df[col].nunique(dropna=False)

        # 1. Constant column
        if n_unique <= 1:
            df = df.drop(columns=[col])
            report["dropped"][col] = "constant"
            continue

        # 2. ID-like column
        if n_unique / n_rows >= id_threshold:
            df = df.drop(columns=[col])
            report["dropped"][col] = "id-like"
            continue

        # 3. Text columns (anything NOT numeric)
        if not pd.api.types.is_numeric_dtype(df[col]):
            # Try datetime detection
            parsed = pd.to_datetime(df[col], errors="coerce")
            if parsed.notna().mean() > 0.9:
                df[f"{col}_year"] = parsed.dt.year
                df[f"{col}_month"] = parsed.dt.month
                df[f"{col}_quarter"] = parsed.dt.quarter
                df[f"{col}_weekday"] = parsed.dt.weekday
                df = df.drop(columns=[col])
                report["datetime_expanded"].append(col)
                continue

            # High-cardinality text -> drop
            if n_unique > cardinality_threshold:
                df = df.drop(columns=[col])
                report["dropped"][col] = f"high-cardinality ({n_unique} unique)"
                continue

    report["final_shape"] = df.shape
    return df, report


# Re-test
df_fe, fe_report = engineer_features(df, target_col="Diagnosis")

print("Dropped columns:")
for col, reason in fe_report["dropped"].items():
    print(f"  {col:<28} -> {reason}")
print("\nDatetime expanded:", fe_report["datetime_expanded"] or "None")
print("Final shape:", fe_report["final_shape"])
print("\nRemaining columns:", df_fe.columns.tolist())

Dropped columns:
  Patient ID                   -> id-like
  Patient Name                 -> high-cardinality (13818 unique)
  Blood Pressure Levels        -> high-cardinality (4458 unique)
  Cholesterol Levels           -> high-cardinality (5952 unique)
  Symptoms                     -> high-cardinality (5786 unique)

Datetime expanded: None
Final shape: (15000, 17)

Remaining columns: ['Age', 'Gender', 'Hypertension', 'Heart Disease', 'Marital Status', 'Work Type', 'Residence Type', 'Average Glucose Level', 'Body Mass Index (BMI)', 'Smoking Status', 'Alcohol Intake', 'Physical Activity', 'Stroke History', 'Family History of Stroke', 'Dietary Habits', 'Stress Levels', 'Diagnosis']


In [20]:


def encode_features(df, target_col, low_card_max=15):
    
    df = df.copy()
    report = {"target_encoded": None, "binary_encoded": [], "onehot_encoded": []}
    encoders = {}

   
    if not pd.api.types.is_numeric_dtype(df[target_col]):
        le = LabelEncoder()
        df[target_col] = le.fit_transform(df[target_col])
        encoders["__target__"] = le
        report["target_encoded"] = dict(zip(le.classes_, le.transform(le.classes_)))

   
    cat_cols = [
        c for c in df.columns
        if c != target_col and not pd.api.types.is_numeric_dtype(df[c])
    ]

    binary_cols = [c for c in cat_cols if df[c].nunique() == 2]
    onehot_cols = [c for c in cat_cols if 2 < df[c].nunique() <= low_card_max]

   
    for col in binary_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        encoders[col] = le
        report["binary_encoded"].append(col)

   
    if onehot_cols:
        df = pd.get_dummies(df, columns=onehot_cols, drop_first=False)
        # convert bool columns (from get_dummies) to int
        bool_cols = df.select_dtypes(include="bool").columns
        df[bool_cols] = df[bool_cols].astype(int)
        report["onehot_encoded"] = onehot_cols

    report["final_shape"] = df.shape
    return df, report, encoders



df_enc, enc_report, encoders = encode_features(df_fe, target_col="Diagnosis")

print("Target encoding:", enc_report["target_encoded"])
print("Binary encoded  :", enc_report["binary_encoded"])
print("One-hot encoded :", enc_report["onehot_encoded"])
print("Final shape     :", enc_report["final_shape"])
print("\nColumns after encoding:")
print(df_enc.columns.tolist())

Target encoding: {'No Stroke': np.int64(0), 'Stroke': np.int64(1)}
Binary encoded  : ['Gender', 'Residence Type', 'Family History of Stroke']
One-hot encoded : ['Marital Status', 'Work Type', 'Smoking Status', 'Alcohol Intake', 'Physical Activity', 'Dietary Habits']
Final shape     : (15000, 35)

Columns after encoding:
['Age', 'Gender', 'Hypertension', 'Heart Disease', 'Residence Type', 'Average Glucose Level', 'Body Mass Index (BMI)', 'Stroke History', 'Family History of Stroke', 'Stress Levels', 'Diagnosis', 'Marital Status_Divorced', 'Marital Status_Married', 'Marital Status_Single', 'Work Type_Government Job', 'Work Type_Never Worked', 'Work Type_Private', 'Work Type_Self-employed', 'Smoking Status_Currently Smokes', 'Smoking Status_Formerly Smoked', 'Smoking Status_Non-smoker', 'Alcohol Intake_Frequent Drinker', 'Alcohol Intake_Never', 'Alcohol Intake_Rarely', 'Alcohol Intake_Social Drinker', 'Physical Activity_High', 'Physical Activity_Low', 'Physical Activity_Moderate', 'Dietar

In [22]:


def scale_features(df, target_col, min_unique=15):
    """Scale continuous numeric features using StandardScaler.

    Binary and one-hot (0/1) columns are skipped. The target is never scaled.
    Returns the scaled DataFrame, a report, and the fitted scaler.
    """
    df = df.copy()

    # Identify continuous columns (numeric, many unique values, not target)
    continuous_cols = [
        c for c in df.columns
        if c != target_col
        and pd.api.types.is_numeric_dtype(df[c])
        and df[c].nunique() >= min_unique
    ]

    report = {"scaled_columns": continuous_cols, "skipped": []}
    report["skipped"] = [
        c for c in df.columns
        if c != target_col and c not in continuous_cols
    ]

    scaler = None
    if continuous_cols:
        scaler = StandardScaler()
        df[continuous_cols] = scaler.fit_transform(df[continuous_cols])

    report["final_shape"] = df.shape
    return df, report, scaler



df_scaled, scale_report, scaler = scale_features(df_enc, target_col="Diagnosis")

print("Scaled columns :", scale_report["scaled_columns"])
print("Skipped (binary/onehot):", len(scale_report["skipped"]), "columns")
print("\nBefore vs After (first continuous column):")
col = scale_report["scaled_columns"][0]
print(f"  {col}: mean={df_scaled[col].mean():.3f}, std={df_scaled[col].std():.3f}")
print("\nSample scaled data:")
print(df_scaled[scale_report["scaled_columns"]].head())

Scaled columns : ['Age', 'Average Glucose Level', 'Body Mass Index (BMI)', 'Stress Levels']
Skipped (binary/onehot): 30 columns

Before vs After (first continuous column):
  Age: mean=0.000, std=1.000

Sample scaled data:
        Age  Average Glucose Level  Body Mass Index (BMI)  Stress Levels
0  0.093263               0.036180              -0.705993      -0.536939
1  1.232733               1.340814               0.704803      -1.146031
2 -1.331076               1.470981              -0.989536       0.796103
3  0.900388               1.379345               0.003554       0.113920
4 -0.144127               1.182983               0.219323       0.632518


In [23]:
from feature_engineering import process_features
from data_cleaning import clean_dataset

# Full pipeline: raw -> clean -> process
df_raw = pd.read_csv(os.path.join(SAMPLE_DIR, "stroke_sliced.csv"))
df_clean, _ = clean_dataset(df_raw)
df_processed, report = process_features(
    df_clean, target_col="Diagnosis",
    artifacts_path="../models/preprocessing_artifacts.pkl"
)

print("Input shape :", report["input_shape"])
print("Final shape :", report["final_shape"])
print("Artifacts   :", report.get("artifacts_path"))

2026-07-02 17:08:52,841 | data_cleaning | INFO | Cleaning started. Input shape: (15000, 22)
2026-07-02 17:08:53,161 | data_cleaning | INFO | Filled 'Symptoms' with mode (16.7% missing)
2026-07-02 17:08:53,470 | data_cleaning | INFO | Removed 0 duplicate rows
2026-07-02 17:08:53,612 | data_cleaning | INFO | Skipped 'Hypertension' for outliers (2 unique values)
2026-07-02 17:08:53,614 | data_cleaning | INFO | Skipped 'Heart Disease' for outliers (2 unique values)
2026-07-02 17:08:53,623 | data_cleaning | INFO | Skipped 'Stroke History' for outliers (2 unique values)
2026-07-02 17:08:53,632 | data_cleaning | INFO | Cleaning finished. Output shape: (15000, 22)
2026-07-02 17:08:53,637 | feature_engineering | INFO | Feature processing started. Input shape: (15000, 22)
2026-07-02 17:08:57,422 | feature_engineering | INFO | Feature engineering: dropped 5 columns
2026-07-02 17:08:57,541 | feature_engineering | INFO | Encoding: 3 binary, 6 one-hot
2026-07-02 17:08:57,594 | feature_engineering | 

Input shape : (15000, 22)
Final shape : (15000, 35)
Artifacts   : ../models/preprocessing_artifacts.pkl
